In [ ]:
import inflect, json, sys, os
import regex as re 
from collections import defaultdict
from ruamel.yaml import YAML
from ruamel.yaml.scalarstring import LiteralScalarString
from datasets import load_dataset

In [ ]:
data_train = load_dataset("schema_guided_dstc8", trust_remote_code=True, split="train")
data_val = load_dataset("schema_guided_dstc8", trust_remote_code=True, split="validation")
data_test = load_dataset("schema_guided_dstc8", trust_remote_code=True, split="test")

In [ ]:
labels = set()
for data in data_train:
    labels.add(data['services'][0])

In [ ]:
labels

In [ ]:
label_set = set(label for label in labels if re.search(r"_1$",label))

In [ ]:
label_set

In [ ]:
def get_user_data(data,i):
    return data['turns']['frames'][i]['slots'][0]['start'],\
            data['turns']['frames'][i]['slots'][0]['exclusive_end'],\
            data['turns']['frames'][i]['slots'][0]['slot'],\
            data['turns']['utterance'][i]
                
def get_system_data(data,i):
    return data['turns']['frames'][i]['slots'][0]['start'],\
            data['turns']['frames'][i]['slots'][0]['exclusive_end'],\
            data['turns']['frames'][i]['actions'][0]['values'],\
            data['turns']['frames'][i]['slots'][0]['slot'],\
            data['turns']['utterance'][i]
                

In [ ]:
def get_user(start_idx, end_idx, slot_val, text):
    zipped = list(zip(start_idx, end_idx, slot_val))
    sorted_list = sorted(zipped, key=lambda x:x[0], reverse=True)
    for start, end, value in sorted_list:
        text = text[:start]+ f"[{text[start:end]}]" +f"({value})"+text[end:]
    return text

def get_sys(start_idx, end_idx, slot_name, slot_val, text):
    zipped = list(zip(start_idx, end_idx, slot_name, slot_val))
    sorted_list = sorted(zipped, key=lambda x:x[0], reverse=True)
    for start, end, name, value in sorted_list:
        text = text[:start]+ f"{{{value}}}"+text[end:]
    return text

In [ ]:
def format_label(label):
    label = label.split('_')[0].lower()
    p = inflect.engine()
    if re.search(r's$',label):
        label = p.singular_noun(label)
    return label

def fetch(data_file, label,  nlu_data, domain_data, slots, stories):
    count=0

    for data in data_file:
        if data['services'][0] == label:
            intent = format_label(data['services'][0])
            utterence = data['turns']['utterance']

            for i in range(0, len(utterence),2):
                usr_strt_idx, usr_end_idx, usr_slot_val, usr_text  = get_user_data(data,i)
                sys_strt_idx, sys_end_idx, sys_slot_name, sys_slot_val, sys_text = get_system_data(data,i+1)
                user = get_user(usr_strt_idx, usr_end_idx, usr_slot_val, usr_text)
                system = get_sys(sys_strt_idx,sys_end_idx,sys_slot_name,sys_slot_val,sys_text)
                nlu_data[intent].append(user)
                domain_data['utter_'+intent].append(system)
                slots.update(usr_slot_val or [])
                slots.update(sys_slot_val or [])
                stories[intent].update(usr_slot_val or [])
                stories[intent].update(sys_slot_val or [])                
                count+=1
            if count>=34:
                return

In [ ]:
nlu_data=defaultdict(list)
domain_data=defaultdict(list)
slots=set()
stories=defaultdict(set)

In [ ]:
for label in label_set:
    fetch(data_train, label, nlu_data, domain_data, slots, stories)

In [ ]:
intents = [format_label(label) for label in label_set]
slot_val = [slot for slot in slots]

In [ ]:
nlu_yaml = {"version": "3.1", "nlu":[]}
domain_yaml = {"version": "3.1", "intent":intents,"slots":{} ,"responses":{}, "entities":slot_val}
stories_yaml = {"version": "3.1", "stories":[]}

In [ ]:
for intent, text in nlu_data.items():
    nlu_format = {
        "intent": intent,
        "examples": LiteralScalarString('\n'.join([f'- {t}' for t in text]) + '\n')
        }
    
    nlu_yaml['nlu'].append(nlu_format)

for intent, text in domain_data.items():
    domain_yaml['responses'][intent]=[{"text": txt} for txt in text]

for slot in slots:
    domain_yaml['slots'][slot]= {
            "type": "text",
            "mappings":[
                        {
                           "type": "from_entity",
                            "entity": slot
                        }
                        ] 
                       }
                       
for story in stories:
    story_format = {
        "story": story+" path",
        "steps":[
            {'intent':'greet'},
            {'action':'utter_greet'},
            {'intent':story},
            {'action':'utter_'+story}
        ]
    }
    stories_yaml['stories'].append(story_format)

yaml = YAML()
yaml.default_flow_style = False
yaml.width = 4096
yaml.indent(mapping=2, sequence=2, offset=2)
yaml.preserve_quotes = True

In [ ]:
yaml.block_seq_indent = 0
with open("/Volumes/LaCie/Projects_portfolio/NLP/SupportIQ/rasa/data/nlu.yml", "w") as f:
    yaml.dump(nlu_yaml, f)

with open("/Volumes/LaCie/Projects_portfolio/NLP/SupportIQ/rasa/data/stories.yml", "w") as f:
    yaml.dump(stories_yaml, f)
    
yaml.block_seq_indent = 2
with open("/Volumes/LaCie/Projects_portfolio/NLP/SupportIQ/rasa/domain.yml", "w") as f:
    yaml.dump(domain_yaml, f)


In [ ]:
def intent_tagging(train_data, stop=10):
    count=0
    for data in train_data:
        # print(data['turns']['frames'][0].keys())
        utterence = data['turns']['utterance']
        for i in range(len(utterence)):
            end_index = data['turns']['frames'][i]['slots'][0]['exclusive_end']
            slot_val = data['turns']['frames'][i]['slots'][0]['slot']
            start_index = data['turns']['frames'][i]['slots'][0]['start']
            slots = data['turns']['frames'][i]['actions'][0]['values']
            print("slot_val", slot_val)
            print(slots)
            print("start_index", start_index, "end_index", end_index)
            print(data['turns']['utterance'][i])
            if len(start_index)>1 and len(slot_val)>1:
                for idx in range(len(start_index)-1):
                    print(data['turns']['utterance'][i][:start_index[idx]] +\
                            # f"[{data['turns']['utterance'][i][start_index[idx]:end_index[idx]]}]"+\
                            f"{{{slot_val[idx]}}}" + \
                            data['turns']['utterance'][i][end_index[idx]:start_index[idx+1]]+ \
                            # f"[{data['turns']['utterance'][i][start_index[idx+1]:end_index[idx+1]]}]" +\
                            f"{{{slot_val[idx+1]}}}"+data['turns']['utterance'][i][end_index[idx+1]:])
            elif len(start_index)>1 and len(slot_val)==1:
                for idx in range(len(start_index)-1):
                    print(data['turns']['utterance'][i][:start_index[idx]] +\
                            # f"[{data['turns']['utterance'][i][start_index[idx]:end_index[idx]]}]"+\
                            f"{{{slot_val[0]}}}" + \
                            data['turns']['utterance'][i][end_index[idx]:start_index[idx+1]]+ \
                            # f"[{data['turns']['utterance'][i][start_index[idx+1]:end_index[idx+1]]}]" +\
                            f"{{{slot_val[0]}}}"+data['turns']['utterance'][i][end_index[idx+1]:])
            elif len(start_index)==1: 
                print(data['turns']['utterance'][i][:start_index[0]] +\
                    # f"[{data['turns']['utterance'][i][start_index[0]:end_index[0]]}]"+\
                    f"{{{slot_val[0]}}}" + \
                    data['turns']['utterance'][i][end_index[0]:])
                
        count+=1
        if count>stop:
            break

In [ ]:
test = "The event will start at 10:30 pm and you need to go for a Movie show."
test[24:32]

slot_value = ['event_time', 'event_name']
start_index = [24, 47]
end_index =  [29, 66]
slot_name = ['Dentist appointment', '10 am']
sentence = "The event will start at 10 am and the event is Dentist appointment."

In [ ]:
data_train['turns']

In [ ]:
slots = list(zip(start_index, end_index, slot_value, slot_name))

In [ ]:
slots = sorted(slots, key=lambda x: x[0], reverse=True)


In [ ]:
for start, end, value, slot_name in slots:
        sentence = sentence[:start] + f"{{{value}}}" + sentence[end:]

In [ ]:
sentence

In [ ]:
for story in stories['movie']:
    print(story)

In [ ]:
slots

In [ ]:
start_idx = [47, 24]
end_idx = [66, 29]
slot_names = [['Dentist appointment'], ['10 am']]
slot_val = ['event_name', 'event_time']
text = "The event will start at 10 am and the event is Dentist appointment."

output = get_sys(start_idx, end_idx, slot_names, slot_val, text)
print(output)


In [ ]:
intent_tagging(calendar_data,100)

In [ ]:
data_train

In [ ]:
def filter_data(dataset, domain):
    return dataset.filter(lambda x: any(domain in service for service in x['services']))

In [ ]:
calendar_data = filter_data(data_train,'Calendar_1')

In [ ]:
calendar_data[0]['turns']

In [ ]:
t=0
i=3
for data in calendar_data:
    # print(data['turns']['frames'][0].keys())
    end_index = data['turns']['frames'][i]['slots'][0]['exclusive_end']
    slot_val = data['turns']['frames'][i]['slots'][0]['slot']
    # slot_val = "#####"
    start_index = data['turns']['frames'][i]['slots'][0]['start']
    slots =data['turns']['frames'][i]['actions'][0]['slot']
    print("]]]]]]",data['turns']['utterance'][i])
    # print(slot_val)
    if len(start_index)>1 and len(slot_val)>1:
        for idx in range(len(start_index)-1):
            print(data['turns']['utterance'][i][:start_index[idx]] +\
                    f"[{data['turns']['utterance'][i][start_index[idx]:end_index[idx]]}]"+\
                    f"({slot_val[idx]})" + \
                    data['turns']['utterance'][i][end_index[idx]:start_index[idx+1]]+ \
                    f"[{data['turns']['utterance'][i][start_index[idx+1]:end_index[idx+1]]}]" +\
                    f"({slot_val[idx+1]})"+data['turns']['utterance'][i][end_index[idx+1]:])
    elif len(start_index)>1 and len(slot_val)==1:
        for idx in range(len(start_index)-1):
            print(data['turns']['utterance'][i][:start_index[idx]] +\
                    f"[{data['turns']['utterance'][i][start_index[idx]:end_index[idx]]}]"+\
                    f"({slot_val[0]})" + \
                    data['turns']['utterance'][i][end_index[idx]:start_index[idx+1]]+ \
                    f"[{data['turns']['utterance'][i][start_index[idx+1]:end_index[idx+1]]}]" +\
                    f"({slot_val[0]})"+data['turns']['utterance'][i][end_index[idx+1]:])
    elif len(start_index)==1: 
        print(data['turns']['utterance'][i][:start_index[0]] +\
            f"[{data['turns']['utterance'][i][start_index[0]:end_index[0]]}]"+\
            f"({slot_val[idx]})" + \
            data['turns']['utterance'][i][end_index[idx]:])
    else:
        print('Missing indexes', start_index)
    print('----------##########----------')
    print(data['turns']['utterance'][i][:])
    print('----------##########----------')
    print(data['turns']['frames'][i]['service'])
    print("Starts with ---->",data['turns']['frames'][i]['slots'][0])
    print(data['turns']['frames'][i]['state'])
    print(data['turns']['frames'][i]['actions'])
    print(data['turns']['frames'][i]['service_results'])
    print(data['turns']['frames'][i]['service_call'])   
    print(slot_val) 
    t+=1
    if t>200:
        break

In [ ]:
(['service', 'slots', 'state', 'actions', 'service_results', 'service_call'])

In [ ]:
import requests
from datetime import datetime
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
api_key = os.getenv("MOVIE_API_KEY")
url = os.getenv("MOVIE_BASE_URL")

In [ ]:
zipcode="76013"
start_date = datetime.today().date()

In [ ]:
payload = {"startDate":start_date, "zip":zipcode, "radius":"20", "api_key":api_key}
headers = {"Content-Type":"application/json"}

In [ ]:
result = requests.get(url,headers=headers, params=payload)

In [ ]:
result

In [ ]:
if result.status_code == 200:
    try:
        response = result.json()
    except ValueError:  # includes JSONDecodeError
        print("Response was not valid JSON")
        response = None
else:
    print(f"Error: {result.status_code}, body: {result.text}")
    response = None

In [ ]:
title_names = set([val.get('title') for val in response])

In [ ]:
title_names

In [ ]:
single_movie_metadata = next((movie for movie in response if movie.get('title')=='Sketch'),None)

In [ ]:
from fuzzywuzzy import process

In [ ]:
best_match, value = process.extractOne("Superman", title_names)

In [ ]:
best_match, value

In [ ]:
[theater for theater in single_movie_metadata.get("showtimes")]

In [ ]:
def get_time(iso_date):
    return datetime.fromisoformat(iso_date).strftime("%H:%M")

single_movie_metadata = next((movie for movie in response if movie.get('title')=='Superman'),None)

In [ ]:
set([theater.get('theatre').get("name") for theater in single_movie_metadata.get('showtimes')])

In [ ]:
theater_name = "Studio Movie Grill Arlington"

In [ ]:
[get_time(theater.get("dateTime")) for theater in single_movie_metadata.get('showtimes') if theater.get('theatre').get("name") == theater_name]

In [ ]:
movie_list = set([theater.get("theatre").get("name") for showtime in response for theater in showtime.get("showtimes")])

In [ ]:
'\n'.join((f". {movie}" for movie in movie_list))

In [ ]:
set([theater.get('theatre').get("name") for theater in response.get('showtimes')])

In [ ]:
response

In [ ]:
zipcode = "76013"
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
zip_api_base = os.getenv("ZIPCODE_API_BASE")
zip_url = f"{zip_api_base}/{zipcode}"

In [ ]:
zip_api = f"http://api.zippopotam.us/us/{zipcode}"
headers = {"Content-Type": "application/json"}
payload = {"zipcode":zipcode}

In [ ]:
requests.get(zip_api,json=payload,headers=headers).status_code

In [ ]:
requests.get(zip_url)

In [ ]:
zip_api_base

In [ ]:
from dateutil import parser
from datetime import timedelta

In [ ]:
parser.parse("Aug 15").date() == datetime.today().date()

In [ ]:
movie_name = "The Fantastic Four: First Steps"
theater_name = "AMC Eastchase 9"
movie_metadata = [movie for movie in response if movie.get('title')==movie_name]

In [ ]:
[showtime.get("dateTime") for movie in movie_metadata for showtime in movie.get("showtimes")
         if showtime.get("theatre").get("name")==theater_name]

In [ ]:
dict_movie = {}

In [ ]:
dict_movie['zipcode'] = movie_metadata

In [ ]:
movie_metadata = dict_movie['zipcode']

In [ ]:
[movie.get("title") for movie in movie_metadata]

In [ ]:
def get_time(iso_date):
        return datetime.fromisoformat(iso_date).strftime("%H:%M")

In [ ]:
[showtime.get("dateTime") for movie in response if movie.get('title')==movie_name 
                                    for showtime in movie.get("showtimes")
                                    if showtime.get("theatre").get("name")==theater_name]

In [ ]:
movie_metadata